# A Simple Calculator Using EBNF

This file shows the implementation of a *symbolic calculator* using `Larkita`.  It is a variant of the notebook
`01-Calculator-larkita.ipynb`:  Instead of left recursion, the grammar uses the repetition operator `*` of the
*extended Backus-Naur form* (EBNF).  The grammar for the language implemented by this parser is as follows:
$$
\begin{array}{lcl}
  \texttt{stmnt}   & \rightarrow & \;\texttt{IDENTIFIER}\; \texttt{':='} \; \;\texttt{expr}\; \texttt{';'} \\
                   & \mid        & \;\texttt{expr}\; \texttt{';'} \\[0.2cm]
  \texttt{expr}    & \rightarrow & \;\texttt{product}\; \bigl((\texttt{'+'} \mid \texttt{'-'})\; \texttt{product}\bigr)^* \\[0.2cm]
  \texttt{product} & \rightarrow & \;\texttt{factor}\; \bigl((\texttt{'*'} \mid \texttt{'/'})\; \texttt{factor}\bigr)^* \\[0.2cm]
  \texttt{factor}  & \rightarrow &   \texttt{'('} \; \texttt{expr} \;\texttt{')'}      \\
                   & \mid        & \;\texttt{NUMBER}                                   \\
                   & \mid        & \;\texttt{IDENTIFIER}
\end{array}
$$
The abstract syntax tree and its evaluation are the same as in `01-Calculator-larkita.ipynb`.

## Setup and Imports
We import the class `Larkita`.  It generates a *canonical LR(1) parser* that returns the abstract syntax tree
of its input as a *nested tuple*.

In [ ]:
from larkita import Larkita

## Specification of the Grammar

`Larkita` reads grammars in the *extended Backus-Naur Form* (EBNF) of `Lark`:  both the lexical tokens and the
grammar rules are defined in a single string.  Let us break down the components of our grammar definition:

* `stmnt`: A statement is either an assignment or an expression followed by a semicolon.  The first rule of the
   grammar defines the start symbol.  As explained in `01-Calculator-larkita.ipynb`, the string literals of an
   alternative form the *head* of the tuple that is returned, so the two alternatives yield
   `((':=', ';'), ('IDENTIFIER', name), ast(expr))` and `(';', ast(expr))`.
* **Repetition** `( ... )*`: The rule `expr: product (ADD_OP product)*` states that an expression is a product that
   is followed by zero or more occurrences of an additive operator and another product.  This single rule replaces
   the three left-recursive rules
   ```
   expr: expr "+" product
       | expr "-" product
       | product
   ```
   of `01-Calculator-larkita.ipynb`.  The rule for `product` is built the same way.  The hierarchy `expr`, `product`,
   `factor` still enforces operator precedence (e.g., multiplication in `product` happens before addition in `expr`).
* **Flat tuples**: The head of a tuple is determined by the grammar alone, not by the input.  As the alternative
   `product (ADD_OP product)*` contains no string literal, the name of the rule is used as head.  The operands and
   the operators are collected in a single *flat* tuple, for example `1 - 2 - 3` yields
   `('expr', p1, ('ADD_OP', '-'), p2, ('ADD_OP', '-'), p3)`, where `p1`, `p2`, and `p3` are the tuples
   representing the products.  Since the head is fixed by the grammar, a single product yields the tuple
   `('expr', p1)`, too.  Hence the grammar no longer specifies that `-` associates to the left.  We take care of
   this when we build the AST.
* **Named Terminals** (`ADD_OP`, `MUL_OP`): The string literals of an alternative only show up in its head and the
   head of the alternative `product (ADD_OP product)*` would be the same for all operators.  Since we need to know
   which operator has been used, we give the operators a name:  Named terminals are children of the tuple.  Note
   that the definition of a terminal may use the alternative `|`, too.
* `%ignore "(", ")"` removes the parentheses from the head of the alternative `"(" expr ")"`.  As this alternative
   then consists of a single symbol, it returns the abstract syntax tree of `expr`.
* **Lexical Imports** (`%import common...`): Instead of manually writing regular expressions for common tokens,
   we import standard terminal definitions.
   * `CNAME` parses standard variable names (letters, numbers, underscores) and is aliased to `IDENTIFIER`.
   * `NUMBER` handles both integers and floating-point numbers.
   * `WS` handles whitespace.
* `%ignore WS`: This directs the lexer to completely discard spaces and tabs so they do not clutter our parsing logic.

In [ ]:
calc_grammar = r"""
stmnt: IDENTIFIER ":=" expr ";"
     | expr ";"

expr: product (ADD_OP product)*

product: factor (MUL_OP factor)*

factor: "(" expr ")"    %ignore "(", ")"
      | NUMBER
      | IDENTIFIER

// Lexical definitions
%import common.CNAME -> IDENTIFIER
%import common.NUMBER
%import common.WS

ADD_OP: "+" | "-"
MUL_OP: "*" | "/"

// Ignore whitespace
%ignore WS
"""

We generate the parser.  `Larkita` always generates a canonical LR(1) parser and writes a description of all
LR(1) states into the file specified by `out_file`.

In [ ]:
parser = Larkita(calc_grammar, out_file='calc-ebnf.out')

The nested tuple returned for `1 - 2 * 3 - 4;` shows the flat tuples:  after the head `'expr'`, operands and
operators alternate.

In [ ]:
parser.parse('1 - 2 * 3 - 4;')

## Generating the Abstract Syntax Tree (ADT)

Our evaluation function expects the *binary* tuples of `01-Calculator-larkita.ipynb`, where every operator has
exactly two operands.  For example, the string `x := y + z;` should be represented by the ADT
`((':=', ';'), ('IDENTIFIER', 'x'), ('+', ('IDENTIFIER', 'y'), ('IDENTIFIER', 'z')))`.

A flat tuple with head `'expr'` or `'product'` has the form `(head, operand, op, operand, op, ..., operand)`.  The
function `to_binary` combines the operands from left to right, so `1 - 2 - 3` becomes
`('-', ('-', ('NUMBER', '1'), ('NUMBER', '2')), ('NUMBER', '3'))`.  The operators are tokens like `('ADD_OP', '-')`,
whose text is exactly the head of the binary tuple.  The tuples representing tokens are returned unchanged, while
all other tuples are converted recursively.  Therefore, the ADT is the same as in `01-Calculator-larkita.ipynb`.

In [ ]:
def to_binary(ast):
    match ast:
        case ('NUMBER' | 'IDENTIFIER', _):
            return ast
        case ('expr' | 'product', first, *rest):
            # Combine the operands of a flat tuple from left to right
            result = to_binary(first)
            for (_, op), operand in zip(rest[0::2], rest[1::2]):
                result = (op, result, to_binary(operand))
            return result
        case (head, *children):
            return (head, *[to_binary(child) for child in children])

In [ ]:
to_binary(parser.parse('1 - 2 * 3 - 4;'))

## Evaluating the Abstract Syntax Tree

Now that we have our pure ADT made of nested tuples, we can write our own recursive evaluation function.  It is the
same function as in `01-Calculator-larkita.ipynb`.  The variable `Names2Values` is a global dictionary that stores
the values associated with the variables we define.

In [ ]:
Names2Values = {}

In [ ]:
def evaluate_adt(adt):
    match adt:
        case ('NUMBER', value):
            return float(value)
        case ('IDENTIFIER', var_name):
            if var_name not in Names2Values:
                print(f"Syntax Error: Variable '{var_name}' is undefined.")
                return 0.0
            return Names2Values[var_name]
        case ((':=', ';'), ('IDENTIFIER', var_name), expr):
            Names2Values[var_name] = evaluate_adt(expr)
        case (';', expr):
            print(evaluate_adt(expr))
        case (('+' | '-' | '*' | '/') as op, left, right):
            val_left  = evaluate_adt(left )
            val_right = evaluate_adt(right)
            match op:
                case '+': return val_left + val_right
                case '-': return val_left - val_right
                case '*': return val_left * val_right
                case '/': return val_left / val_right
        case _:
            raise ValueError(f"Unknown ADT structure: {adt}")

## Execution

The REPL (Read-Eval-Print Loop) takes the input string, parses it into a nested tuple, converts the flat tuples into
binary tuples, and then evaluates the result.

In [ ]:
def main():
    while True:
        try:
            s = input('calc> ')
            if s == '':
                break
            # 1. Parse the string into a nested tuple
            ast = parser.parse(s)
            # 2. Convert the flat tuples into binary tuples
            adt = to_binary(ast)
            # 3. Evaluate the nested tuple
            evaluate_adt(adt)
        except (EOFError, NotImplementedError):  # no input possible
            break
        except Exception as e:
            print(f"Error: {e}")

In [ ]:
main()

Check the populated memory map:

In [ ]:
Names2Values

We can even get the parser states:  `Larkita` has written them into the file `calc-ebnf.out`.  As LR parsing works
with plain BNF rules, the repetition `(ADD_OP product)*` is translated into the auxiliary left-recursive rules
```
__expr_star_0: ADD_OP product
             | __expr_star_0 ADD_OP product
```
and `expr: product __expr_star_0 | product`, which show up in the states.  An auxiliary rule does not build a tuple
of its own.  Instead, it collects its children in a list, and these children are spliced into the tuple of `expr`.
This is why the tuple is flat.  In the list of rules at the beginning of the file, the notation `*__expr_star_0`
stands for these spliced children.

In [ ]:
with open('calc-ebnf.out') as handle:
    print(handle.read())